# 🚀 Hướng Dẫn Chạy CrowdDiff Trên Kaggle Notebook

Chào mừng bạn đến với notebook chạy dự án **CrowdDiff** trên Kaggle. Kaggle cho phép bạn truy cập GPU xịn (như P100 hoặc T4x2) lên tới 30 giờ mỗi tuần hoàn toàn miễn phí!

**Yêu cầu trước khi bắt đầu:**
1. Bạn đã có tài khoản Kaggle.
2. Bạn đã upload file `crowddiff_colab.zip` dưới dạng một **Dataset** trên Kaggle (Ví dụ đặt tên là `crowddiff-source`)
3. Đã nhấn nút **Add Data** ở thanh menu bên phải và thêm dữ liệu `crowddiff-source` của bạn vào notebook này.

### Bước 1: Kiểm tra GPU và Cấu Hình
Ở thanh công cụ bên phải (Session Options), hãy kiểm tra **ACCELERATOR** đã chọn là **GPU P100** hoặc **GPU T4x2** chưa. Máy chủ Kaggle rất mạnh.

In [ ]:
!nvidia-smi

### Bước 2: Chép dữ liệu Mã Nguồn sang Bộ Nhớ Làm Việc (Working Directory)
Thư mục `/kaggle/input/` chỉ cho phép đọc, nên chúng ta cần chép code sang thư mục `/kaggle/working/` để có thể thao tác và lưu file.

In [ ]:
import os

input_dir = '/kaggle/input'
found_code = False

# Xoá thư mục cũ nếu có lỗi để chép lại từ đầu
!rm -rf /kaggle/working/crowddiff-main
!mkdir -p /kaggle/working/crowddiff-main

# Tự động quét để tìm thư mục chứa code
for root, dirs, files in os.walk(input_dir):
    if 'run_pipeline.sh' in files: # Nhận diện thư mục chứa code
        folder_path = root
        print(f"🔎 Đã TỰ ĐỘNG tìm thấy mã nguồn tại: {folder_path}")
        
        # Chép toàn bộ dữ liệu ở thư mục đó sang working
        !cp -r "{folder_path}/"* /kaggle/working/crowddiff-main/
        found_code = True
        break

if found_code:
    os.chdir('/kaggle/working/crowddiff-main')
    print("✅ Đã sao chép thành công!")
    print("Thư mục hiện tại:", os.getcwd())
else:
    print("❌ LỖI: Vẫn không tìm thấy mã nguồn của bạn!")

### Bước 3: Cài đặt thư viện và Fix các đường dẫn cũ
Xóa bỏ các tham chiếu cài đặt cục bộ có trong requirements.txt và tải các gói.

In [ ]:
!pip install blobfile mpi4py einops kagglehub

### Bước 4: Tự động tải ShanghaiTech và Tiền xử lý dữ liệu
Ở Kaggle, Internet rất nhanh nên bước download của kagglehub chỉ mất vài chục giây!

In [ ]:
import os

# 1. Tự động đi tìm bộ dữ liệu gốc ShanghaiTech mà bạn đã thêm ở cột bên phải
dataset_path = ""
for root, dirs, files in os.walk('/kaggle/input'):
    if 'part_A' in dirs:  # Tìm dấu hiệu của bộ dataset
        dataset_path = root
        break

if not dataset_path:
    print("❌ Lỗi: Không thể tìm thấy thư mục ShanghaiTech!")
else:
    print(f"✅ Đã tìm thấy dữ liệu gốc tại: {dataset_path}")
    
    # 2. Tự động sửa mã nguồn run_pipeline.sh để trỏ thẳng tới đây
    with open('run_pipeline.sh', 'r') as f:
        script = f.read()
    
    # Sửa đường dẫn KAGGLE_DATA_DIR
    import re
    script = re.sub(r'KAGGLE_DATA_DIR=.*', f'KAGGLE_DATA_DIR="{dataset_path}"', script)
    
    # Vá lỗi python script
    script = script.replace('PYTHON_CMD=.venv/bin/python', 'PYTHON_CMD=python3')
    script = script.replace('.venv/bin/python', 'python3')
    
    with open('run_pipeline.sh', 'w') as f:
        f.write(script)
        
    print("✅ Đã cập nhật đường dẫn mới thành công!")
    print("🚀 Đang tự động tiến hành lại Tiền Xử Lý (Bước 4), vui lòng đợi vài phút...")
    
    # 3. Chạy bước preprocess
    !chmod +x run_pipeline.sh
    !bash run_pipeline.sh --steps preprocess

### Bước 5: Bắt Đầu Huấn Luyện! (Train)
Kaggle thường ổn định hơn Colab rất nhiều. Hãy để máy chạy. 
Ngoài ra, nút `Save Version` (góc phải trên cùng) cho phép chạy Notebook *ngầm* trên server máy chủ (Run in background) liên tục 12h, bạn hoàn toàn có thể tắt trình duyệt máy tính đi ngủ!

In [ ]:
!bash run_pipeline.sh --steps train

### Bước 6: Lưu Checkpoint (Quan trọng)
Kaggle lưu thư mục working cho bạn ở một mức độ nào đó, nhưng để an toàn bạn có thể xuất các thư mục kết quả thành file zip thành output (sẽ nằm ở thanh công cụ Output bên phải dưới cùng) để dễ dàng tải về máy tính sau khi Training.

In [ ]:
import os
# Nén toàn bộ thư mục kết quả thành file train_results.zip 
!zip -r /kaggle/working/train_results.zip /kaggle/working/crowddiff-main/results
print("Đã nén xong! Hãy xem màn hình bên phải phần Data > Output để tải file zip này về.")